# 🎙️ HooperTTS + Qwen3-TTS

An open-source narration compiler for expressive AI voice generation.

Supported

• Voice Cloning
• Gaming News
• Documentary
• YouTube Shorts
• Podcast

GitHub:
https://github.com/XADITYAM/HooperTTS

First Run

The first execution downloads the Qwen3-TTS model (~4.5 GB).

This usually takes 2–5 minutes.

Subsequent generations in the same Colab session are much faster.

In [ ]:
import os

# Always pull the latest code, even if this runtime already has the repo
# cloned from an earlier cell run in this session (previously this only
# cloned once and silently kept re-running stale code on every retest).
if not os.path.exists("/content/HooperTTS"):
    !git clone https://github.com/XADITYAM/HooperTTS.git
else:
    !cd /content/HooperTTS && git pull

%cd /content/HooperTTS

In [ ]:
# Installs HooperTTS plus the optional [enhancement] extra (transformers, accelerate)
# so the Script Enhancement dropdown in the app works, and requirements.txt (gradio),
# which app.py needs but isn't pulled in automatically.
!pip install -e ".[enhancement]"
!pip install -r requirements.txt

In [ ]:
%cd /content

if not os.path.exists("/content/Qwen3-TTS"):
    !git clone https://github.com/QwenLM/Qwen3-TTS.git

%cd /content/Qwen3-TTS

In [ ]:
!pip install -e .

# soundfile is the only one of these HooperTTS actually uses (qwen/runner.py writes
# WAV output with it). faster-whisper/ctranslate2/sentencex/pysrt were for subtitle
# generation, but core/subtitles.py is still an empty stub with no callers, so they're
# dropped here to save install time. Add them back if you build that feature out.
!pip install soundfile

In [ ]:
# --- Permanent fix for the Xet/CAS 403 SignatureError ---
# Must run BEFORE anything imports huggingface_hub (qwen_tts does this internally).
import os

!pip uninstall -y hf-xet -q

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "30"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"

print("Xet disabled, plain HTTP downloads forced.")

In [ ]:
import torch

print("="*40)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print(f"Free VRAM: {free_bytes / 1024**3:.1f} GiB / {total_bytes / 1024**3:.1f} GiB")
    print("Use this to decide Quality (needs ~4.5 GiB free) vs Fast (~2 GiB free)")
    print("for Script Enhancement once the app is running.")

print(torch.__version__)
print("="*40)

In [ ]:
import qwen_tts

print("✓ qwen_tts imported successfully")

In [ ]:
import time
from huggingface_hub import snapshot_download

def robust_snapshot_download(repo_id, max_retries=5, backoff_seconds=10, **kwargs):
    """Retries snapshot_download with backoff to ride out transient HF CDN errors
    (e.g. 403 SignatureError from the xet-bridge CDN, connection resets, etc.).
    """
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            return snapshot_download(repo_id=repo_id, **kwargs)
        except Exception as e:
            last_err = e
            wait = backoff_seconds * attempt
            print(f"[attempt {attempt}/{max_retries}] download failed: {e}")
            if attempt < max_retries:
                print(f"Retrying in {wait}s...")
                time.sleep(wait)
    raise last_err

model_path = robust_snapshot_download(
    repo_id="Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    max_workers=1,
)

print(model_path)

In [ ]:
!hoopertts doctor

## Optional: Script Enhancement

The app now has a **Script Enhancement** dropdown (off by default) and a model-tier
dropdown:

- **Off (optimize only)** — current default behavior, no LLM involved.
- **Enhance script** / **Enhance + optimize** — loads a small Qwen3 model to rewrite
  pacing/clarity before narration, validates that no facts changed, then releases the
  model before Qwen3-TTS loads.
- **Quality (Qwen3-1.7B)** vs **Fast (Qwen3-0.6B)** — pick Fast if the free VRAM printed
  above is tight (under ~4.5 GiB) after the TTS model is also loaded.

If enhancement fails (e.g. out of memory), the app falls back to the original script
and shows why in the Diagnostics box — it won't crash the run.

In [ ]:


%cd /content/HooperTTS

!python app.py